In [ ]:
import cv2
import mediapipe as mp
from feat import Detector
from PIL import Image
import tempfile, os

# Initialize PyFeat
detector = Detector(
    face_model="retinaface",
    landmark_model="mobilefacenet",
    au_model="xgb",
    emotion_model="resmasknet"
)

# Mapping: MediaPipe 468 -> dlib-style 68 face landmarks
DLIB_68_IDXS = [
    127, 234, 93, 132, 58, 172, 150, 176, 152, 400, 379, 378, 365, 397, 288, 361, 323, 454,
    70, 63, 105, 66, 107, 336, 296, 334, 293, 300, 383, 353, 372, 340, 346, 280, 352,
    33, 160, 158, 133, 153, 144,
    362, 385, 387, 263, 373, 380,
    168, 6, 197, 195, 5, 4, 75, 97, 2, 326, 305, 294, 278, 331, 279, 429, 358,
    61, 40, 37, 0, 267, 270, 409, 291, 375, 321, 405, 314, 17, 84, 181, 91, 146,
    78, 81, 13, 311, 402, 14, 178, 87, 317
]

# Pose landmarks (upper body only)
POSE_IDX = [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]  # shoulders, elbows, wrists, hands

mp_holistic = mp.solutions.holistic

# Function to extract FAUs using PyFeat (optimized with temp file)
def extract_aus_from_frame(frame):
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmpfile:
        tmp_path = tmpfile.name
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        Image.fromarray(rgb_frame).save(tmp_path)

    feat_result = detector.detect_image(tmp_path)
    os.remove(tmp_path)

    if feat_result.empty:
        return None, None
    else:
        aus = feat_result.aus.iloc[0].to_dict()
        emotions = feat_result.emotions.iloc[0].to_dict()
        return aus, emotions

# Capture video
cap = cv2.VideoCapture(0)

# For FAU optimization
frame_count = 0
last_aus, last_emotions = None, None
FAU_INTERVAL = 10  # run FAU detection every 10 frames

with mp_holistic.Holistic(
    static_image_mode=False,
    model_complexity=1,
    enable_segmentation=False,
    refine_face_landmarks=True
) as holistic:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb)

        # ===== FACE: 68 Landmarks =====
        if results.face_landmarks:
            for idx in DLIB_68_IDXS:
                lm = results.face_landmarks.landmark[idx]
                x, y = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (x, y), 1, (0, 255, 0), -1)

        # ===== POSE: Arms + Shoulders =====
        if results.pose_landmarks:
            for idx in POSE_IDX:
                lm = results.pose_landmarks.landmark[idx]
                x, y = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (x, y), 4, (0, 0, 255), -1)

        # ===== HANDS =====
        if results.left_hand_landmarks:
            for lm in results.left_hand_landmarks.landmark:
                x, y = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (x, y), 2, (255, 0, 0), -1)

        if results.right_hand_landmarks:
            for lm in results.right_hand_landmarks.landmark:
                x, y = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (x, y), 2, (0, 255, 255), -1)

        # ===== FAU (every N frames) =====
        if frame_count % FAU_INTERVAL == 0:
            last_aus, last_emotions = extract_aus_from_frame(frame)

        if last_aus is not None:
            y_offset = 20
            for au_name, au_val in last_aus.items():
                cv2.putText(frame, f"{au_name}: {au_val:.2f}", (10, y_offset),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
                y_offset += 15

            if last_emotions:
                cv2.putText(frame, f"Emotion: {max(last_emotions, key=last_emotions.get)}",
                            (10, y_offset + 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        frame_count += 1

        # ===== Display =====
        cv2.imshow("Holistic + 68 Landmarks + Hands + FAU", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()


100%|██████████| 1/1 [00:00<00:00,  1.03it/s]
